In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Layer 1: Extreme ESI Feature Engineered Logistic Regressor (`models/lr_feng_extreme.ipynb`)

This notebook trains a **Layer 1 Multinomial Logistic Regressor** using Feature Engineered inputs, patient demographics, and chief complaint indicator:
- **Target Output**: Predicts whether a patient is **ESI 1** (Highest Acuity), **ESI 5** (Lowest Acuity), or **neither** (Intermediate ESI 2, 3, 4).
- **Feature Set (13 Predictors)**: `age`, `gender`, `cc_breathingdifficulty`, and the **10 Clinical Feature Engineering flags** defined in `TODO.md` (`is_dyspnea_total`, `is_dyspnea_moderate`, `is_bradypnea`, `is_tachypnea`, `is_hypotension`, `is_hypertension`, `is_bradycardia_total`, `is_bradycardia_moderate`, `is_tachycardia_total`, `is_tachycardia_moderate`).
- **Configurable Target Class Subsampling**:
  - `keep_ratio_1 = 1.00` (Keeps 100% of ESI 1 rows).
  - `keep_ratio_5 = 1.00` (Keeps 100% of ESI 5 rows).
  - `keep_ratio_neither = 0.10` (Keeps 10% of 'neither' rows, reducing by 90%).
- **NA Handling**: Retains all rows (`na.omit` removed); missing vital signs generate 0 for binary clinical flags without dropping data.
- **Evaluation Metrics**: **Accuracy**, **Multi-Class ROC-AUC**, **Log Loss**, **Target Class Count Comparison**, and **Confusion Matrix**.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
library(jsonlite)
library(caret)
library(nnet)
library(dplyr)
library(ggplot2)
library(pROC)

config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}

config <- fromJSON(config_path)

cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Val Size:        ", config$training$val_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Data, Compute 10 FE Flags + Age, Gender, CC_Breathingdifficulty & Apply Class Subsampling
# ---------------------------------------------------------
set.seed(config$training$random_state)

data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}

cat("Loading dataset from:", data_file, "...\n")

data_env <- new.env()
load(data_file, envir = data_env)

df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
cat(sprintf("Selected main dataset object: '%s' (%d rows)\n", data_obj_name, max(df_sizes)))

raw_df <- get(data_obj_name, envir = data_env)
target_col <- config$classes$target_col

gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0

# Compute 10 Clinical Feature Engineering flags + Age + Gender + cc_breathingdifficulty
df_feng <- data.frame(
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  is_dyspnea_total        = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 < 90, 1, 0),
  is_dyspnea_moderate     = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 >= 90 & raw_df$triage_vital_o2 < 94, 1, 0),
  is_bradypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr < 10, 1, 0),
  is_tachypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr > 30, 1, 0),
  is_hypotension          = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp <= 90, 1, 0),
  is_hypertension         = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp > 220, 1, 0),
  is_bradycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr < 40, 1, 0),
  is_bradycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr >= 40 & raw_df$triage_vital_hr < 60, 1, 0),
  is_tachycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 150, 1, 0),
  is_tachycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 100 & raw_df$triage_vital_hr <= 150, 1, 0)
)

df_feng[[target_col]] <- raw_df[[target_col]]

# Create Layer 1 Target: '1', '5', or 'neither'
raw_esi <- as.character(df_feng[[target_col]])
df_feng$target_layer1 <- factor(ifelse(raw_esi == "1", "1",
                                 ifelse(raw_esi == "5", "5", "neither")),
                                levels = c("1", "5", "neither"))

# ---------------------------------------------------------
# Subsample Target Classes (Configurable keep ratios per class)
# ---------------------------------------------------------
keep_ratio_1       <- 1.00  # Keep 100% of ESI 1 rows (configurable)
keep_ratio_5       <- 1.00  # Keep 100% of ESI 5 rows (configurable)
keep_ratio_neither <- 0.05  # Keep 10% of 'neither' rows (reduces 'neither' by 90%)

idx_1       <- which(df_feng$target_layer1 == "1")
idx_5       <- which(df_feng$target_layer1 == "5")
idx_neither <- which(df_feng$target_layer1 == "neither")

kept_1       <- sample(idx_1,       size = round(length(idx_1)       * keep_ratio_1))
kept_5       <- sample(idx_5,       size = round(length(idx_5)       * keep_ratio_5))
kept_neither <- sample(idx_neither, size = round(length(idx_neither) * keep_ratio_neither))

df_feng <- df_feng[sort(c(kept_1, kept_5, kept_neither)), ]

cat(sprintf("Layer 1 FE Dataset Ready (Class Ratios: ESI 1=%.0f%%, ESI 5=%.0f%%, Neither=%.0f%%): %d rows x %d cols\n",
            keep_ratio_1 * 100, keep_ratio_5 * 100, keep_ratio_neither * 100, nrow(df_feng), ncol(df_feng)))
cat("Feature Engineered Input Columns (13):\n", paste(setdiff(names(df_feng), c(target_col, "target_layer1")), collapse = ", "), "\n")
cat("Layer 1 Target Distribution:\n")
print(table(df_feng$target_layer1))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Stratified Data Partitioning & Continuous Feature Scaling
# ---------------------------------------------------------
set.seed(config$training$random_state)

test_size <- config$training$test_size
val_size  <- config$training$val_size

# Stratified Test split (15%)
in_train_val <- createDataPartition(df_feng$target_layer1, p = 1 - test_size, list = FALSE)
train_val_df <- df_feng[in_train_val, ]
test_df      <- df_feng[-in_train_val, ]

# Stratified Validation split (15%)
rel_val_size <- val_size / (1 - test_size)
in_train    <- createDataPartition(train_val_df$target_layer1, p = 1 - rel_val_size, list = FALSE)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]

# Standardize continuous feature (age strictly)
cont_cols <- "age"
preproc <- preProcess(train_df[, cont_cols, drop = FALSE], method = c("center", "scale"))

train_df <- predict(preproc, train_df)
val_df   <- predict(preproc, val_df)
test_df  <- predict(preproc, test_df)

cat(sprintf("Partition sizes:\n  Train: %d rows\n  Val:   %d rows\n  Test:  %d rows\n",
            nrow(train_df), nrow(val_df), nrow(test_df)))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Train Layer 1 Feature Engineered Multinomial Logistic Regression Model
# ---------------------------------------------------------
set.seed(config$training$random_state)

feat_names <- setdiff(names(train_df), c(target_col, "target_layer1"))
formula_lr <- as.formula(paste("target_layer1 ~", paste(feat_names, collapse = " + ")))

cat("Training Layer 1 Feature Engineered Multinomial Logistic Regressor...\n")
lr_feng_model <- multinom(formula_lr, data = train_df, trace = FALSE, MaxNWts = 5000)

cat("Layer 1 Feature Engineered Logistic Regression training complete!\n")
print(summary(lr_feng_model))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Evaluate Scoring Metrics (Accuracy, ROC-AUC, Log Loss & Target Class Counts)
# ---------------------------------------------------------
calc_log_loss <- function(actual_factor, prob_matrix, eps = 1e-15) {
  prob_matrix <- pmax(pmin(prob_matrix, 1 - eps), eps)
  prob_matrix <- prob_matrix / rowSums(prob_matrix)
  classes <- colnames(prob_matrix)
  N <- length(actual_factor)
  
  log_probs <- numeric(N)
  for (i in 1:N) {
    act_cls <- as.character(actual_factor[i])
    if (act_cls %in% classes) {
      log_probs[i] <- log(prob_matrix[i, act_cls])
    } else {
      log_probs[i] <- log(eps)
    }
  }
  return(-mean(log_probs))
}

evaluate_layer1_lr <- function(model, data, set_name) {
  prob_matrix <- predict(model, newdata = data, type = "probs")
  target_classes <- levels(data$target_layer1)
  
  max_idx <- max.col(prob_matrix, ties.method = "first")
  pred_factor <- factor(colnames(prob_matrix)[max_idx], levels = target_classes)
  actual_factor <- factor(data$target_layer1, levels = target_classes)
  
  cm <- confusionMatrix(pred_factor, actual_factor)
  acc <- as.numeric(cm$overall["Accuracy"])
  
  roc_auc <- tryCatch({
    as.numeric(pROC::multiclass.roc(actual_factor, prob_matrix)$auc)
  }, error = function(e) NA)
  
  log_loss <- calc_log_loss(actual_factor, prob_matrix)
  
  cat(sprintf("============================================================\n"))
  cat(sprintf("   LAYER 1 FEATURE ENGINEERED LR - %s SET BENCHMARK\n", toupper(set_name)))
  cat(sprintf("============================================================\n"))
  cat(sprintf("  Accuracy   : %.4f (%.2f%%)\n", acc, acc * 100))
  cat(sprintf("  ROC-AUC    : %.4f\n", roc_auc))
  cat(sprintf("  Log Loss   : %.4f\n", log_loss))
  cat("\nTarget Class Counts (Actual vs Predicted Comparison):\n")
  class_counts_df <- data.frame(
    Class = target_classes,
    Actual_Count = as.numeric(table(actual_factor)[target_classes]),
    Predicted_Count = as.numeric(table(pred_factor)[target_classes]),
    Diff = as.numeric(table(pred_factor)[target_classes]) - as.numeric(table(actual_factor)[target_classes])
  )
  print(class_counts_df)
  cat("\nConfusion Matrix (Rows: Predicted, Columns: Actual):\n")
  print(cm$table)
  cat(sprintf("============================================================\n\n"))
}

# Benchmark on Validation and Test Sets
evaluate_layer1_lr(lr_feng_model, val_df, "Validation")
evaluate_layer1_lr(lr_feng_model, test_df, "Test")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Save Layer 1 Feature Engineered Model Artifacts
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)

model_path <- file.path(deploy_dir, "lr_feng_extreme_model.rds")
saveRDS(list(model = lr_feng_model, preproc = preproc), file = model_path)
cat("Layer 1 Feature Engineered Logistic Regressor model saved to:", model_path, "\n")